In [1]:
#We are working on the recruitment demo problem

#Step 1 -lets all install the necessary packages
!pip install -q langchain langchain-google-genai langchain-community python-dotenv pypdf docx2txt langchain_core langchain-text-splitters langchain-classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.0/346.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [8]:
import os
from google.colab import files, userdata
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_core.output_parsers import StrOutputParser

#class API key
os.environ['GOOGLE_API_KEY'] =userdata.get('GOOGLE_API_KEY')

#setup LLM and output Parser
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.3)
output_parser = StrOutputParser()

# Define -prompt
template= """
You are an AI Resume Screener

Compare the following resume with the job description

Job Description:
{job_desc}

Resume Text:
{resume_text}

Give a simple analysis with :
-Fit Score (0-100)
-Domain Relevant Knowledge
-Top 5 matching skills
-Missing important skills
-One-line Verdict

"""

prompt=PromptTemplate(template=template,input_variables=["job_desc","resume_text"],output_parser=output_parser)

###Step 6 - Upload and Extract Resume Text

print("Please upload your resume file (PDF/DOCX/TXT):")

uploaded=files.upload()

if not uploaded:
  raise ValueError("No Files uploaded.")

resume_file=list(uploaded.keys())[0]

#Detect file type and load
if resume_file.lower().endswith('.pdf'):
    loader=PyPDFLoader(resume_file)
elif resume_file.lower().endswith('.docx') or resume_file.lower().endswith('.doc'):
    loader=Docx2txtLoader(resume_file)
else:
    loader=TextLoader(resume_file,encoding="utf-8")
doc=loader.load()


#Split larget text into chunks

splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks=splitter.split_documents(doc)
resume_text=" ".join(c.page_content for c in chunks).strip()

#Safety Fallback condition

if not resume_text:
   raise ValueError("No resume text found.")

#Run the Agent and give job description

job_description = """We are hiring a senior AI engineer who is skilled in Python, SQL, Cloud(AWS/GCP/Azure), Data Analysis and Langchain."""

chain=LLMChain(llm=llm,prompt=prompt,output_parser=output_parser)
result=chain.run(job_desc=job_description,resume_text=resume_text)
print(result)

Please upload your resume file (PDF/DOCX/TXT):


Saving Anurag_Gupta.pdf to Anurag_Gupta.pdf


/tmp/ipykernel_11848/1368736824.py:76: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain=LLMChain(llm=llm,prompt=prompt,output_parser=output_parser)
/tmp/ipykernel_11848/1368736824.py:77: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result=chain.run(job_desc=job_description,resume_text=resume_text)


Here's an analysis of the resume against the job description:

**Fit Score:** 97/100

**Domain Relevant Knowledge:**
The candidate possesses extensive and highly relevant domain knowledge in Applied AI/LLM Systems, Agentic AI, LangChain/LangGraph, RAG pipelines, NLP, prompt engineering, and AI evaluation methodologies. Their experience in regulated financial services (JPMorgan Chase) demonstrates an ability to build robust, scalable, and reliable production AI systems, which is crucial for a senior role. The projects listed further solidify their practical application of these advanced AI concepts.

**Top 5 Matching Skills:**

1.  **Langchain:** Explicitly mentioned multiple times, including advanced usage with LangGraph, ReAct pattern, and integration into complex projects (FinSight, ConvFinQA).
2.  **Python:** Demonstrated as a core skill for building production-grade data pipelines, FastAPI applications, and AI systems, with specific frameworks like PySpark and Pydantic.
3.  **Cloud